In [0]:
from pyspark.sql.functions import *
from framework.reader import *
from framework.writer import *
from framework.audit import *
from framework.validator import *
from framework.metrics import *

In [0]:
erp=read_json(spark, "s3://retail-lakehouse-ashu/raw/erp/erp_products.json")

In [0]:
clean_erp_df,corrupt_erp_df,corrupt_erp_count=separate_corrupt_records(erp)

In [0]:
clean_erp_df.printSchema()

In [0]:
valid_condition = (
    col("product_id").isNotNull() &
    col("product_name").isNotNull() &
    col("category.category_id").isNotNull() &
    col("category.category_name").isNotNull() &
    col("supplier.supplier_id").isNotNull() &
    col("supplier.supplier_name").isNotNull() &
    col("pricing.cost_price").isNotNull() &
    col("pricing.selling_price").isNotNull() &
    (col("pricing.cost_price") > 0) &
    (col("pricing.selling_price") > 0) &
    col("status").isin("ACTIVE", "INACTIVE") &
    col("created_date").isNotNull()
    )

In [0]:
good_erp_df, bad_erp_df=split_records(clean_erp_df,valid_condition)

In [0]:
bronze_erp=add_audit_columns_clean(good_erp_df,"erp",1)

In [0]:
bronze_erp.display()

In [0]:
total_records,good_records,bad_records=calculate_metrics(erp,good_erp_df,bad_erp_df)
print(f"Total Records   : {total_records}")
print(f"Good Records    : {good_records}")
print(f"Bad Records     : {bad_records}")

In [0]:
duplicate_df=erp.groupBy(col("product_id")).count().where(col("count")>1)
display(duplicate_df)

In [0]:
assert total_records==good_records+bad_records

In [0]:
write_delta(bronze_erp,"s3://retail-lakehouse-ashu/bronze/erp/","overwrite")

In [0]:
bronze_bad_df=combine_bad_records(bad_erp_df,corrupt_erp_df,corrupt_erp_count,1,"Bronze_erp","erp","erp_products")

In [0]:
write_delta(bronze_bad_df,"s3://retail-lakehouse-ashu/bad_records/","append")